In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
dbutils.widgets.text('externalfiles_zone', 'None', '07-Externalfiles Zone')
externalfiles_zone = dbutils.widgets.get('externalfiles_zone')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_ext_fasttrack_extfile(
    spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema=(
        #    "ExtFastTrackId string,"
        #    "SourceSystem string,"
        #    "PortfolioKey string,"
        #    "PortfolioName string,"
        #    "IngestionTimestamp timestamp,"
        #    "BatchId string,"
        #    "ExtFastTrackKey string,"
        #    "EntityVersion string,"
        #    "ExposureKey string,"
        #    "PartyKey string,"
        #    "GrantAccount string,"
        #    "PositiveForInstallmentsExtensionReduction string,"
        #    "BankNumber string,"
        #    "RestructuringAmount string,"
        #    "ProgramIndicatorProsarmozo string,"
        #    "EffectiveDateOfNewInstallments string,"
        #    "ProsarmozoDuration string,"
        #    "ProsarmozoAmount string,"
        #    "AccruedInterestChargeIndicator string,"
        #    "RestructuringChannel string,"
        #    "InterestRatesChange string,"
        #    "InterestRateAPeriod string,"
        #    "SpreasAPeriod string,"
        #    "DurationAPeriod string,"
        #    "InterestRateBPeriod string,"
        #    "SpreasBPeriod string,"
        #    "PeriodType string,"
        #    "PeriodNumber string,"
        #    "StandardDate string,"
        #    "PeriodChanged string,"
        #    "ExtensionInstallments string,"
        #    "FastTrackFlag string,"
        #    "SubmitDate string,"
        #    "Status string,"
        #    "ErrorDescription string,"
        #    "Timestamp string,"
        #    "FileType string,"
        #    "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ExtFastTrack"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "extfile",
            )        
        else:
            logging.info("Tables do not exist, no need to delete entries for business date!")

        # ================== fasttrack ==================

        fasttrack_extra_filters = None

        if fasttrack_extra_filters is not None:
            fasttrack_extra_filters.append(date_filter)
            fasttrack_final_filters = " AND ".join(fasttrack_extra_filters)
        else:
            fasttrack_final_filters = date_filter

        fasttrack_selected_fields = [
            "ExposureKey",
            "CDI",
            "GrantAccount",
            "PositiveForInstallmentsExtensionReduction",
            "BankNumber",
            "RestructuringAmount",
            "ProgramIndicatorProsarmozo",
            "EffectiveDateOfNewInstallments",
            "ProsarmozoDuration",
            "ProsarmozoAmount",
            "AccruedInterestChargeIndicator",
            "RestructuringChannel",
            "InterestRatesChange",
            "InterestRateAPeriod",
            "SpreasAPeriod",
            "DurationAPeriod",
            "InterestRateBPeriod",
            "SpreasBPeriod",
            "PeriodType",
            "PeriodNumber",
            "StandardDate",
            "PeriodChanged",
            "ExtensionInstallments",
            "FastTrackFlag",
            "SubmitDate",
            "Status",
            "ErrorDescription",
            "TIMESTAMP",
            # "FileType",
            "BusinessDate",
        ]

        fasttrack = load_landed_delta_table(
            spark,
            externalfiles_zone,  # type: ignore
            "extfasttrack",
            fasttrack_final_filters,
            fasttrack_selected_fields,
        )

        fasttrack = fasttrack.withColumnRenamed(
            "PositiveForInstallmentsExtensionReduction",
            "PositiveForInstallmentsExtensionReduction",
        )
        # ======================== result_df_ ========================

        primary_key_fields_id = [
            fasttrack.ExposureKey,
            fasttrack.CDI,
            fasttrack.Status,
            lit("extfile"),
        ]

        primary_key_fields_key = [
            fasttrack.ExposureKey,
            fasttrack.CDI,
            fasttrack.Status,
        ]

        target_table = fasttrack.withColumn(
            "ExtFastTrackId", sha2(concat_ws(";", *primary_key_fields_id), 256)
        ).withColumn("ExtFastTrackKey", concat_ws(";", *primary_key_fields_key))

        result_df = target_table.select(
            target_table.ExtFastTrackId,
            lit("extfile").alias("SourceSystem"),
            lit("-9999").cast("string").alias("PortfolioKey"),
            lit("Core Retail").cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            target_table.ExtFastTrackKey.alias("ExtFastTrackKey"),
            lit(None).alias("EntityVersion"),
            target_table.ExposureKey.alias("ExposureKey"),
            target_table.CDI.alias("PartyKey"),
            target_table.GrantAccount.alias("GrantAccount"),
            target_table.PositiveForInstallmentsExtensionReduction.alias(
                "PositiveForInstallmentsExtensionReduction"
            ),
            target_table.BankNumber.alias("BankNumber"),
            target_table.RestructuringAmount.alias("RestructuringAmount"),
            target_table.ProgramIndicatorProsarmozo.alias("ProgramIndicatorProsarmozo"),
            target_table.EffectiveDateOfNewInstallments.alias(
                "EffectiveDateOfNewInstallments"
            ),
            target_table.ProsarmozoDuration.alias("ProsarmozoDuration"),
            target_table.ProsarmozoAmount.alias("ProsarmozoAmount"),
            target_table.AccruedInterestChargeIndicator.alias(
                "AccruedInterestChargeIndicator"
            ),
            target_table.RestructuringChannel.alias("RestructuringChannel"),
            target_table.InterestRatesChange.alias("InterestRatesChange"),
            target_table.InterestRateAPeriod.alias("InterestRateAPeriod"),
            target_table.SpreasAPeriod.alias("SpreasAPeriod"),
            target_table.DurationAPeriod.alias("DurationAPeriod"),
            lit(None).alias("InterestRateBPeriod"),
            target_table.SpreasBPeriod.alias("SpreasBPeriod"),
            target_table.PeriodType.alias("PeriodType"),
            target_table.PeriodNumber.alias("PeriodNumber"),
            target_table.StandardDate.alias("StandardDate"),
            target_table.PeriodChanged.alias("PeriodChanged"),
            target_table.ExtensionInstallments.alias("ExtensionInstallments"),
            target_table.FastTrackFlag.alias("FastTrackFlag"),
            target_table.SubmitDate.alias("SubmitDate"),
            target_table.Status.alias("Status"),
            target_table.ErrorDescription.alias("ErrorDescription"),
            target_table.TIMESTAMP.alias("Timestamp"),
            # target_table.FileType.alias("FileType"),
            target_table.BusinessDate.alias("BusinessDate"),
        )
        # ========================== Append ==========================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                externalfiles_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_ext_fasttrack_extfile(spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix)